##### Importação de Pacotes

In [0]:
from pyspark.sql.functions import col, regexp_replace, initcap
from pyspark.sql.types import StringType

##### Diretórios nos Catálogos

In [0]:
path_bronze_ibge_dtb = "/Volumes/01_bronze/schemas/ibge/dtb_territorio/"
path_silver_ibge_dtb = "/Volumes/02_silver/schemas/ibge/"

##### Arquivos na camada Bronze

In [0]:
# Carrega arquivo *.csv no DataFrame
df_dtb = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .option("encoding", "UTF-8")
    .option("inferSchema", "true")
    .csv(f"{path_bronze_ibge_dtb}/IBGE_DTB_Brasil_Relatorio.csv")
)

##### Padronização no DataFrame

In [0]:
# Limpa os campos com caracteres especiais / Somente primeiras letras em Maiúsculas
for field in df_dtb.schema.fields:
    if isinstance(field.dataType, StringType):
        df_dtb = df_dtb.withColumn(
            field.name,
            initcap(
                regexp_replace(col(field.name), "_", " ")
            )
        )

##### Carrega arquivos (formato: Parquet) na camada: Silver

In [0]:
# Salva arquivo Parquet na camada: Silver
df_dtb.write.format("parquet") \
    .mode("overwrite") \
    .save(f"{path_silver_ibge_dtb}/dtb_territorio")

##### Fim